# 05 — Ablation Studies (5-Step Sequential)

Each step varies one axis, holds everything else constant.
Results from each step carry forward as the fixed value for subsequent steps.

1. **Step 1 — Input Representation:** Feature layers (L0–L3), spatial/temporal, graph structure
2. **Step 2 — Encoder Architecture:** ST-GCN vs Transformer vs LSTM vs 1D-CNN
3. **Step 3 — Pre-Training Regime:** Random init vs SimCLR vs SimCLR + Aux (RQ2)
4. **Step 4 — Few-Shot Method:** ProtoNet vs k-NN vs Linear probe
5. **Step 5 — K-Shot Sensitivity:** K = 1, 3, 5, 8, 10, 15

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from src.config import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Ablation Runner Helper

Reusable function that takes a config, runs the full pipeline,
and returns results dict.

In [ ]:
from src.data.graph_builder import GraphBuilder
from src.data.dataset import FineBadmintonDataset, ShuttleSetDataset, EpisodicSampler
from src.models.stgcn_model import STGCN
from src.models.transformer_encoder import SkeletonTransformer
from src.models.proto_net import PrototypicalNetwork
from sklearn.metrics import f1_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression


def build_encoder(cfg, adjacency):
    """Build encoder based on config."""
    feature_dim = FEATURE_DIMS[cfg.ablation.feature_layer]
    
    if cfg.ablation.encoder == 'transformer':
        return SkeletonTransformer(
            in_channels=feature_dim * cfg.stgcn.num_nodes,
            d_model=cfg.transformer.d_model,
            nhead=cfg.transformer.nhead,
            num_layers=cfg.transformer.num_layers,
            embedding_dim=cfg.transformer.embedding_dim,
        )
    else:
        return STGCN(
            in_channels=feature_dim,
            num_nodes=cfg.stgcn.num_nodes,
            adjacency=adjacency,
            embedding_dim=cfg.stgcn.embedding_dim,
        )


def extract_embeddings(encoder, dataset, device):
    """Extract embeddings for all samples in dataset."""
    encoder.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for i in range(len(dataset)):
            x, y = dataset[i]
            x = x.unsqueeze(0).to(device)
            emb = encoder(x).cpu().numpy()[0]
            embeddings.append(emb)
            labels.append(y)
    return np.array(embeddings), np.array(labels)


def evaluate_classifier(embeddings, labels, splits, method='protonet', knn_k=5):
    """Evaluate a classifier across folds."""
    fold_results = []
    
    for train_idx, val_idx, test_idx in splits:
        X_train = embeddings[train_idx]
        y_train = labels[train_idx]
        X_test = embeddings[test_idx]
        y_test = labels[test_idx]
        
        if method == 'protonet':
            # Compute prototypes (class centroids)
            prototypes = {}
            for c in np.unique(y_train):
                prototypes[c] = X_train[y_train == c].mean(axis=0)
            
            # Classify by nearest centroid
            preds = []
            for x in X_test:
                dists = {c: np.linalg.norm(x - p) for c, p in prototypes.items()}
                preds.append(min(dists, key=dists.get))
            preds = np.array(preds)
            
        elif method == 'knn':
            clf = KNeighborsClassifier(n_neighbors=knn_k)
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
            
        elif method == 'linear_probe':
            clf = LogisticRegression(max_iter=1000, C=1.0)
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
        
        macro_f1 = f1_score(y_test, preds, average='macro')
        per_class_f1 = f1_score(y_test, preds, average=None)
        fold_results.append({'macro_f1': macro_f1, 'per_class_f1': per_class_f1})
    
    return fold_results


def summarize_results(fold_results):
    """Aggregate fold results."""
    f1s = [r['macro_f1'] for r in fold_results]
    return {
        'macro_f1_mean': np.mean(f1s),
        'macro_f1_std': np.std(f1s),
        'fold_f1s': f1s,
    }

---
## Step 1 — Input Representation

Fix encoder = ST-GCN, SSL + Aux pre-training.
Vary: feature layers (L0→L1→L2→L3), spatial/temporal, graph structure.

In [ ]:
step1_configs = {
    'L0_raw_xy': get_config('L0', **{'ablation.feature_layer': 'L0'}),
    'L1_kinematics': get_config('L1', **{'ablation.feature_layer': 'L1'}),
    'L2_court_ctx': get_config('L2', **{'ablation.feature_layer': 'L2'}),
    'L3_joint_angles': get_config('L3', **{'ablation.feature_layer': 'L3'}),
    'spatial_only': get_config('spatial', **{'ablation.use_temporal': False}),
    'temporal_only': get_config('temporal', **{'ablation.use_spatial': False}),
    'single_player': get_config('single', **{'ablation.single_player': True}),
    'no_inter_edges': get_config('no_inter', **{'ablation.use_inter_player': False}),
}

print(f"Step 1 variants: {len(step1_configs)}")
for name, cfg in step1_configs.items():
    fl = cfg.ablation.feature_layer
    print(f"  {name}: {fl} ({FEATURE_DIMS[fl]}-dim)")

In [ ]:
# TODO: Run Step 1 ablations
# For each config:
# 1. Load SSL-pretrained encoder for that feature layer
# 2. Load FineBadminton dataset with matching feature layer
# 3. Extract embeddings
# 4. Evaluate with ProtoNet
# 5. Store results

step1_results = {}
# for name, cfg in step1_configs.items():
#     ...
print("Step 1 — pending SSL pre-training for each feature layer")

---
## Step 2 — Encoder Architecture

Fix input = best from Step 1, SSL + Aux pre-training.
Vary: ST-GCN vs Transformer vs LSTM vs 1D-CNN.

In [ ]:
step2_configs = {
    'stgcn': get_config('stgcn'),
    'transformer': get_config('transformer', **{'ablation.encoder': 'transformer'}),
    'lstm': get_config('lstm', **{'ablation.encoder': 'lstm'}),
    'cnn1d': get_config('cnn1d', **{'ablation.encoder': 'cnn1d'}),
}

print(f"Step 2 variants: {len(step2_configs)}")
for name in step2_configs:
    print(f"  {name}")

In [ ]:
step2_results = {}
print("Step 2 — pending Step 1 completion")

---
## Step 3 — Pre-Training Regime (RQ2)

Fix encoder + input = best from Steps 1-2.
Vary: random init vs SimCLR vs SimCLR + Aux.

In [ ]:
step3_configs = {
    'random_init': get_config('random', **{
        'ablation.use_pretrained': False,
    }),
    'ssl_contrastive_only': get_config('ssl_only', **{
        'ablation.use_auxiliary_task': False,
    }),
    'ssl_plus_aux': get_config('ssl_aux'),
}

print(f"Step 3 variants: {len(step3_configs)}")
for name in step3_configs:
    print(f"  {name}")

In [ ]:
step3_results = {}
print("Step 3 — pending Steps 1-2 completion")

---
## Step 4 — Few-Shot Method

Fix encoder + input + pre-training = best from Steps 1-3.
Vary: ProtoNet vs k-NN vs Linear probe.

In [ ]:
step4_configs = {
    'protonet': {'method': 'protonet'},
    'knn_3': {'method': 'knn', 'knn_k': 3},
    'knn_5': {'method': 'knn', 'knn_k': 5},
    'linear_probe': {'method': 'linear_probe'},
}

print(f"Step 4 variants: {len(step4_configs)}")
for name in step4_configs:
    print(f"  {name}")

In [ ]:
step4_results = {}
print("Step 4 — pending Steps 1-3 completion")

---
## Step 5 — K-Shot Sensitivity

Fix all components = best from Steps 1-4.
Vary K from 1 to 15.

In [ ]:
k_values = [1, 3, 5, 8, 10, 15]
step5_results = {}
print("Step 5 — pending Steps 1-4 completion")

---
## Save All Results

In [ ]:
all_results = {
    'step1_input': step1_results,
    'step2_encoder': step2_results,
    'step3_pretraining': step3_results,
    'step4_classifier': step4_results,
    'step5_kshot': step5_results,
}

# Save as JSON (convert numpy arrays to lists)
# with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
#     json.dump(all_results, f, indent=2, default=str)
print("Results saving — pending ablation runs")